# Megaline - Mobile Plan Recommendation

> **Note:** Megaline is a fictional mobile operator created exclusively for academic and portfolio purposes. This notebook was developed as part of a Data Science training program and is intended solely to demonstrate technical skills in machine learning and data analysis.

---

## 1. Introduction and Objective

Megaline is dissatisfied with the fact that many customers are still using legacy plans. The company wants a model that analyses client behaviour and automatically recommends one of its two newest plans: **Smart** or **Ultra**.

The objective of this project is to develop a machine learning classification model capable of analyzing customer behavior (such as call duration, text messages, and mobile data usage) to accurately recommend the most suitable new plan for each user. 

The model is trained on a dataset of users who have already migrated to the new plans. The success criterion for this classification task is an **Accuracy of at least 0.75** on the held-out test set.

The pipeline follows these key steps:
1. Data loading and initial exploration
2. Data splitting (Training, Validation, and Test sets)
3. Model investigation and hyperparameter tuning
4. Final evaluation on the test set
5. Sanity check (Baseline comparison)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading and Initial Exploration

This step covers loading the dataset and performing a preliminary inspection to ensure the data is clean, properly formatted, and ready for modeling — with no missing values or incorrect data types.

In [2]:
df = pd.read_csv('../data/users_behavior.csv')
display(df.head())
df.info()

# Check class balance
print("\nTarget distribution:")
print(df['is_ultra'].value_counts(normalize=True))

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB

Target distribution:
is_ultra
0    0.693528
1    0.306472
Name: proportion, dtype: float64


The dataset contains no missing values and all columns have appropriate numerical types. The target classes are modestly imbalanced: approximately 69% of customers chose Smart (0) and 31% chose Ultra (1). This is not a severe imbalance, but we will pay attention to the baseline performance.

## 3. Data Splitting

Since the platform does not provide a separate, pre-established test set, the source dataset must be divided into three distinct subsets to ensure a robust evaluation process:
- **Training Set (60%)**: Used to teach the model.
- **Validation Set (20%)**: Used to iteratively tune hyperparameters.
- **Test Set (20%)**: Held out completely to evaluate the final model's true accuracy.

First, the features (`X`) are separated from the target variable (`y`), which in this case is the `is_ultra` column.

In [3]:
X = df.drop(['is_ultra'], axis=1)
y = df['is_ultra']

# Splitting 60% for training and 40% for the rest
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=12345)

# Splitting the remaining 40% into 20% for validation and 20% for testing
X_valid, X_test, y_valid, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=12345)

print(f"Training set size: {X_train.shape[0]} rows")
print(f"Validation set size: {X_valid.shape[0]} rows")
print(f"Test set size: {X_test.shape[0]} rows")

Training set size: 1928 rows
Validation set size: 643 rows
Test set size: 643 rows


## 4. Model Investigation and Tuning

In this section, three different classification algorithms are evaluated. Each model is trained on the training set and iteratively adjusted using the validation set to identify the optimal hyperparameter configuration.

### 4.1. Decision Tree

The Decision Tree classifier is tested first. The `max_depth` hyperparameter is iterated over to find the sweet spot between underfitting and overfitting.

In [4]:
best_tree_model = None
best_tree_result = 0
best_depth = 0

for depth in range(1, 11):
    model_tree = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model_tree.fit(X_train, y_train)
    predictions_valid = model_tree.predict(X_valid)
    result = accuracy_score(y_valid, predictions_valid)

    if result > best_tree_result:
        best_tree_model = model_tree
        best_tree_result = result
        best_depth = depth

print(f"Best Decision Tree accuracy: {best_tree_result:.4f} with max_depth = {best_depth}")

Best Decision Tree accuracy: 0.7854 with max_depth = 3


### 4.2. Random Forest

The Random Forest ensemble typically yields higher accuracy and stability by averaging multiple decision trees, though it is computationally more expensive. Both the number of trees (`n_estimators`) and the maximum depth (`max_depth`) are systematically tested.

In [5]:
best_forest_model = None
best_forest_result = 0
best_est = 0
best_forest_depth = 0

for est in range(10, 51, 10):
    for depth in range(1, 11):
        model_forest = RandomForestClassifier(random_state=12345, n_estimators=est, max_depth=depth)
        model_forest.fit(X_train, y_train)
        predictions_valid = model_forest.predict(X_valid)
        result = accuracy_score(y_valid, predictions_valid)

        if result > best_forest_result:
            best_forest_model = model_forest
            best_forest_result = result
            best_est = est
            best_forest_depth = depth

print(f"Best Random Forest accuracy: {best_forest_result:.4f}")
print(f"Optimal hyperparameters: n_estimators = {best_est}, max_depth = {best_forest_depth}")

Best Random Forest accuracy: 0.8087
Optimal hyperparameters: n_estimators = 40, max_depth = 8


### 4.3. Logistic Regression

Finally, a Logistic Regression algorithm is tested to establish a baseline for linear model performance.

In [6]:
model_lr = LogisticRegression(random_state=12345, solver='lbfgs', max_iter=1000)
model_lr.fit(X_train, y_train)
predictions_valid = model_lr.predict(X_valid)
result_lr = accuracy_score(y_valid, predictions_valid)

print(f"Logistic Regression accuracy: {result_lr:.4f}")

Logistic Regression accuracy: 0.7558


## 5. Final Evaluation on the Test Set

Among the evaluated algorithms, the **Random Forest** achieved the best performance on the validation set. This superiority can be attributed to two main factors:

1. **Overcoming Overfitting:** A single Decision Tree has a strong tendency to create highly specific rules, effectively memorizing the training data (overfitting). Random Forest resolves this by building dozens of independent trees using random subsets of data. By taking a majority vote across all trees, the model corrects individual biases, resulting in highly stable predictions on unseen data.
2. **Capturing Non-Linear Patterns:** Logistic Regression attempts to draw a straight mathematical boundary to separate Smart from Ultra users. However, human behavioral data (such as the ratio between megabytes used and call duration) is rarely perfectly linear. Random Forests excel at mapping complex, non-linear relationships natively without requiring manual feature transformations.

With its optimal architecture identified, the Random Forest model is now evaluated on the held-out test set to definitively confirm if it meets the project's strict 0.75 accuracy requirement.

In [7]:
# Utilizing the best Random Forest model saved previously
test_predictions = best_forest_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final accuracy on the test set: {test_accuracy:.4f}")

if test_accuracy >= 0.75:
    print("Success: The model surpassed the minimum required accuracy threshold.")
else:
    print("Failure: The model did not meet the threshold. Further hyperparameter review is required.")

Final accuracy on the test set: 0.7963
Success: The model surpassed the minimum required accuracy threshold.


## 6. Sanity Check (Baseline Comparison)

To ensure the machine learning model has genuinely learned underlying behavioral patterns—and is not merely guessing based on class distribution—it must be compared against a baseline, naive model (the Dummy Classifier).

If a "dumb" model simply predicts the most frequent class (the plan that the majority of users already have) for every customer, what would its accuracy be? The Random Forest must significantly outperform this baseline to prove true predictive intelligence.

In [8]:
# Checking the class distribution in the dataset
print("Class distribution in the dataset:")
print(df['is_ultra'].value_counts(normalize=True))

# Creating the naive baseline model 
dummy_model = DummyClassifier(strategy='most_frequent', random_state=12345)
dummy_model.fit(X_train, y_train)

dummy_predictions = dummy_model.predict(X_test)
dummy_accuracy = accuracy_score(y_test, dummy_predictions)

print("\n----------------------------------------------------")
print(f"Dummy Model accuracy: {dummy_accuracy:.4f}")
print(f"Intelligent Model (Random Forest) accuracy: {test_accuracy:.4f}")
print("----------------------------------------------------")

if test_accuracy > dummy_accuracy:
    print("\nThe model passed the sanity check! It performs considerably better than simply guessing the most frequent class.")
else:
    print("\nThe model failed the sanity check. It does not outperform basic probability.")

Class distribution in the dataset:
is_ultra
0    0.693528
1    0.306472
Name: proportion, dtype: float64

----------------------------------------------------
Dummy Model accuracy: 0.6843
Intelligent Model (Random Forest) accuracy: 0.7963
----------------------------------------------------

The model passed the sanity check! It performs considerably better than simply guessing the most frequent class.


## 7. Conclusion

The objective of this project was to provide Megaline with a reliable tool to recommend the Smart or Ultra plans based on current user behavior. 

After robust testing, the final model — a `RandomForestClassifier` — successfully captured the underlying data patterns and proved its predictive value against a naive baseline guess. The key results are summarized below:

| Metric | Target | Result |
|---|---|---|
| Test Accuracy | ≥ 0.75 | **0.7963** ✅ |
| Dummy Baseline | Benchmark | **0.6843** |

The Random Forest comfortably exceeded both the baseline probability and the project's strict 0.75 accuracy threshold. With an accuracy approaching 80%, this model is statistically robust and ready to be utilized by Megaline for highly targeted and effective plan recommendation campaigns.